In [8]:
#DUPLICATED
from pathlib import Path
import numpy as np
import sys
# Cell 0 — CONFIG
if sys.platform == "win32":
    project_root = Path(r"F:/MScPROJECT_LOCAL")
else:
    project_root = Path("/workspaces/MScPROJECT_LOCAL")
case_name = "07_Tropea_cafe"

case_dir     = project_root / "Data" / case_name
source_dir   = case_dir / "010_source"
frames_for_recon   = case_dir / "019_CUT3R_Frames"

#TEMP
#This needs to be filed after user input decides on what to analyse
ppt_template = '01_test_02'
# video loading
# Auto-find the video in 010_source
videos = list(source_dir.glob("*.mp4"))
assert len(videos) == 1, f"Expected 1 video, found {len(videos)}: {videos}"
video_path = videos[0]
print(f"Case    : {case_name}")
print(f"Video   : {video_path.name}")

Case    : 07_Tropea_cafe
Video   : VID_20260530_113822990.mp4


In [9]:
#TEMP - don't need this as it's a copy
#stabilisation - basic warps
import subprocess

bypass_stab = False  #skipper

src = video_path
stab_dir = case_dir / "011_stabilised"
stab_dir.mkdir(exist_ok=True)
cfr  = stab_dir / f"{case_name}_cfr.mp4"
trf  = stab_dir / f"{case_name}.trf"
out  = stab_dir / f"{case_name}_stab.mp4"
ffmpeg = r"C:\ffmpeg\bin\ffmpeg.exe" if sys.platform == "win32" else "/usr/bin/ffmpeg"

# Parameters
fps        = 30 # frame rate to encode
crf        = 18 # quality
shakiness  = 10 # size of match window
accuracy   = 15 # iterations on matcing and cam solve
smoothing  = 20 # gaussian smoothing on cam path
zoom       = 5 # crop and zoom factor
if not bypass_stab:
    # CFR conversion
    subprocess.run([ffmpeg, "-y" , "-i", str(src), "-vf", f"fps={fps}", "-c:v", "libx264", "-crf", str(crf), "-c:a", "copy", str(cfr)], check=True)

    # Pass 1
    subprocess.run([ffmpeg, "-y" ,"-i", str(cfr), "-vf", f"vidstabdetect=shakiness={shakiness}:accuracy={accuracy}:result={trf}", "-f", "null", "-"], check=True)


ffmpeg version 4.3.9-0+deb11u2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 10 (Debian 10.2.1-6)
  configuration: --prefix=/usr --extra-version=0+deb11u2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-avresample --disable-filter=resample --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librsvg --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab 

In [10]:
from C_Scoring import frame_scoring, sequence_scoring, score_formula

In [11]:
trf = case_dir / "011_stabilised" / "transforms_cfr.trf"
frame_scores = frame_scoring(video_path, frame_range = 'all', ROIs=None)
sequence_scores = sequence_scoring(trf, frame_scores,30)
#final_scores = score_formula(sequence_scores)
#default_cam, per_frame_pdf, edit = 


setting up for frame scoring
scoring frame 0


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:3845: error: (-215:Assertion failed) !dsize.empty() in function 'resize'


In [ ]:
import matplotlib.pyplot as plt
import math

def plot_seq_scores(seq_scores):
    frame_idxs = sorted(seq_scores.keys())
    keys = list(next(iter(seq_scores.values())).keys())

    ncols = 2
    nrows = math.ceil(len(keys) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows), sharex=True)
    axes = axes.flatten()

    for ax, key in zip(axes, keys):
        ax.plot(frame_idxs, [seq_scores[i][key] for i in frame_idxs])
        ax.set_title(key)
        ax.set_xlabel("Frame")

    for ax in axes[len(keys):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

plot_seq_scores(seq_scores)
